# Lab03 – Phân tích khám phá dữ liệu (EDA)
## House Prices – Advanced Regression Techniques (Kaggle) / Ames Housing (De Cock, 2011)

| Thành viên | MSSV | Phần phụ trách |
|---|---|---|
| ... | ... | ... |

**Cấu trúc notebook**
0. Xác định bài toán (từ paper + Kaggle)
1. Tổng quan dữ liệu & xác định tính chất biến
2. Phân tích đơn biến
3. Phân tích đa biến
4. Giá trị thiếu & mẫu bất thường
5. Ngoại lệ (outliers)
6. Tổng kết & xuất dữ liệu đã làm sạch

## 0. Xác định bài toán

- **Nguồn dữ liệu**: De Cock (2011) thu thập từ Văn phòng Thẩm định Ames, Iowa: 2930 giao dịch nhà ở (2006–2010), 80 biến giải thích (23 nominal, 23 ordinal, 14 discrete, 20 continuous). Kaggle dùng bản con: **train 1460 dòng**, test 1459 dòng, 79 biến giải thích + `Id` (không có `SalePrice` ở test).
- **Loại bài toán**: học có giám sát – **hồi quy** (regression) dự đoán biến mục tiêu liên tục `SalePrice` (USD).
- **Thước đo Kaggle**: RMSE giữa log(giá dự đoán) và log(giá thực) → ta cần quan tâm tới phân phối lệch phải của `SalePrice` và phép biến đổi log.
- **Lưu ý từ paper cần kiểm chứng trong EDA**:
  1. Nên loại các căn có `GrLivArea` > 4000 sq ft (outliers, trong đó có các giao dịch *Partial* không phản ánh giá thị trường).
  2. Phương sai tăng theo giá (heteroscedasticity) → cân nhắc biến đổi căn/log.
  3. Biến thứ bậc (ordinal) và biến rời rạc có thể cho hệ số "phi logic" khi coi là categorical.
  4. Khoảng 80% biến thiên giá giải thích được chỉ bằng `Neighborhood` + tổng diện tích.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 11})
pd.set_option("display.max_columns", 100, "display.width", 200)

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
def save(name):
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

CANDIDATES = ["/kaggle/input/house-prices-advanced-regression-techniques/train.csv",
              "train.csv", "data/train.csv", "../input/house-prices-advanced-regression-techniques/train.csv"]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert path, "Không tìm thấy train.csv – hãy đặt file cạnh notebook hoặc chạy trên Kaggle."

## 1. Tổng quan dữ liệu & xác định tính chất biến
> Lưu ý: dữ liệu dùng chuỗi `"NA"` (và cả chuỗi `"None"` ở `MasVnrType`). Nếu đọc bằng `pd.read_csv` mặc định, pandas biến **cả hai** thành NaN → `MasVnrType` bị báo thiếu 872 dòng thay vì 8. Notebook chỉ coi `"NA"` là NaN. Trong dữ liệu này, nhiều `NA` thực chất nghĩa là **"không có tiện ích đó"** (vd. không có hồ bơi) chứ không phải thiếu dữ liệu → sẽ phân biệt ở mục 4.

In [ ]:
# Chỉ coi "NA" là thiếu. KHÔNG dùng mặc định của pandas vì nó biến cả chuỗi "None" (MasVnrType = không ốp gạch) thành NaN.
df = pd.read_csv(path, keep_default_na=False, na_values=["NA", ""])
print(f"Đường dẫn: {path}")
print(f"Kích thước: {df.shape[0]} dòng × {df.shape[1]} cột")
print("Id trùng:", df["Id"].duplicated().sum(), "| Dòng trùng (bỏ Id):", df.drop(columns="Id").duplicated().sum())
print("Bộ nhớ: %.2f MB" % (df.memory_usage(deep=True).sum() / 1e6))
df.head()

In [ ]:
TARGET = "SalePrice"

# Bảng mã hoá thứ tự cho biến ordinal (NaN → "None" = không có tiện ích)
Q = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ORD_MAPS = {
    **{c: Q for c in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
                      "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]},
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Functional": {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8},
    "LotShape": {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
    "LandSlope": {"Sev": 1, "Mod": 2, "Gtl": 3},
    "PavedDrive": {"N": 0, "P": 1, "Y": 2},
    "Fence": {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "Utilities": {"ELO": 1, "NoSeWa": 2, "NoSewr": 3, "AllPub": 4},
}
ORDINAL_CAT = list(ORD_MAPS)                       # ordinal dạng chữ
ORDINAL_NUM = ["OverallQual", "OverallCond"]       # ordinal dạng số 1–10
DISCRETE = ["BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr", "KitchenAbvGr",
            "TotRmsAbvGrd", "Fireplaces", "GarageCars", "YearBuilt", "YearRemodAdd", "GarageYrBlt",
            "MoSold", "YrSold"]
NOMINAL = [c for c in df.select_dtypes("object").columns if c not in ORDINAL_CAT] + ["MSSubClass"]
CONTINUOUS = [c for c in df.select_dtypes(np.number).columns
              if c not in DISCRETE + ORDINAL_NUM + ["Id", "MSSubClass", TARGET]]

type_tbl = pd.DataFrame({
    "Nhóm": ["Continuous (liên tục)", "Discrete (rời rạc/đếm/năm)", "Ordinal (thứ bậc)", "Nominal (định danh)", "Target", "Định danh dòng"],
    "Số biến": [len(CONTINUOUS), len(DISCRETE), len(ORDINAL_NUM) + len(ORDINAL_CAT), len(NOMINAL), 1, 1],
    "Ví dụ": [", ".join(CONTINUOUS[:3]), ", ".join(DISCRETE[:3]), ", ".join((ORDINAL_NUM + ORDINAL_CAT)[:3]),
              ", ".join(NOMINAL[:3]), TARGET, "Id"],
})
covered = set(CONTINUOUS + DISCRETE + ORDINAL_NUM + ORDINAL_CAT + NOMINAL + [TARGET, "Id"])
print("Cột chưa phân loại:", set(df.columns) - covered, "| tổng cột:", len(covered))
type_tbl

In [ ]:
# Chuẩn bị 2 phiên bản dữ liệu phục vụ phân tích
df_cat = df.copy()                                  # NaN ở biến chữ → "None" (không có tiện ích)
for c in df.select_dtypes("object").columns:
    df_cat[c] = df_cat[c].fillna("None")
df_enc = df_cat.copy()                              # ordinal chữ → số thứ tự
for c, m in ORD_MAPS.items():
    df_enc[c] = df_cat[c].map(m)
print("Giá trị ordinal không map được:", int(df_enc[ORDINAL_CAT].isna().sum().sum()))

NUM_ALL = CONTINUOUS + DISCRETE + ORDINAL_NUM
print("Số biến định lượng:", len(NUM_ALL), "| biến phân loại:", len(NOMINAL) + len(ORDINAL_CAT))

## 2. Phân tích đơn biến
### 2.1 Biến mục tiêu `SalePrice`

In [ ]:
y = df[TARGET]
print(y.describe().round(1).to_string())
print(f"\nSkewness = {y.skew():.3f} | Kurtosis = {y.kurt():.3f}")
print(f"Skewness sau log1p = {np.log1p(y).skew():.3f} | Kurtosis = {np.log1p(y).kurt():.3f}")
print("D'Agostino normality test – gốc: p = %.2e | log: p = %.2e" % (stats.normaltest(y)[1], stats.normaltest(np.log1p(y))[1]))

fig, ax = plt.subplots(2, 3, figsize=(16, 8))
for r, (s, nm) in enumerate([(y, "SalePrice"), (np.log1p(y), "log1p(SalePrice)")]):
    sns.histplot(s, kde=True, ax=ax[r, 0], color="#4C72B0")
    ax[r, 0].axvline(s.mean(), c="r", ls="--", label="mean"); ax[r, 0].axvline(s.median(), c="g", ls="--", label="median")
    ax[r, 0].legend(); ax[r, 0].set_title(f"Histogram – {nm}")
    stats.probplot(s, plot=ax[r, 1]); ax[r, 1].set_title(f"Q-Q plot – {nm}")
    sns.boxplot(x=s, ax=ax[r, 2], color="#55A868"); ax[r, 2].set_title(f"Boxplot – {nm}")
save("01_target_distribution")
plt.show()

### 2.2 Biến định lượng: thống kê mô tả, độ lệch, tỷ lệ giá trị 0

In [ ]:
desc = df[NUM_ALL].describe().T
desc["skew"] = df[NUM_ALL].skew()
desc["kurt"] = df[NUM_ALL].kurt()
desc["%zero"] = (df[NUM_ALL] == 0).mean() * 100
desc["%NaN"] = df[NUM_ALL].isna().mean() * 100
desc = desc.round(2)
desc.sort_values("skew", ascending=False)

In [ ]:
n, nc = len(CONTINUOUS), 4
nr = int(np.ceil(n / nc))
fig, axes = plt.subplots(nr, nc, figsize=(16, 3.2 * nr)); axes = axes.ravel()
for a, c in zip(axes, CONTINUOUS):
    sns.histplot(df[c].dropna(), bins=40, ax=a, color="#4C72B0")
    a.set_title(f"{c} (skew={df[c].skew():.2f}, 0s={100*(df[c]==0).mean():.0f}%)"); a.set_xlabel("")
for a in axes[n:]: a.axis("off")
plt.suptitle("Biến liên tục – phân phối", y=1.002, fontweight="bold")
save("02_continuous_hist"); plt.show()

In [ ]:
cols = DISCRETE + ORDINAL_NUM
n, nc = len(cols), 4
nr = int(np.ceil(n / nc))
fig, axes = plt.subplots(nr, nc, figsize=(16, 3.0 * nr)); axes = axes.ravel()
for a, c in zip(axes, cols):
    sns.histplot(df[c].dropna(), discrete=True, ax=a, color="#C44E52"); a.set_title(c); a.set_xlabel("")
for a in axes[n:]: a.axis("off")
plt.suptitle("Biến rời rạc & ordinal dạng số – phân phối", y=1.002, fontweight="bold")
save("03_discrete_hist"); plt.show()

**Biến đổi cho biến lệch phải** (|skew| > 0.75): so sánh skew trước/sau `log1p`.

In [ ]:
skewed = [c for c in CONTINUOUS if df[c].min() >= 0 and abs(df[c].skew()) > 0.75]
tr = pd.DataFrame({"skew gốc": df[skewed].skew(), "skew log1p": np.log1p(df[skewed]).skew(), "%zero": (df[skewed] == 0).mean() * 100}).round(2)
tr["Nhận xét"] = np.where(tr["%zero"] > 50, "Zero-inflated → tạo cờ Has_X + log phần >0", np.where(tr["skew log1p"].abs() < 0.75, "log1p hiệu quả", "log1p cải thiện một phần"))
tr.sort_values("skew gốc", ascending=False)

### 2.3 Biến phân loại: tần suất, số mức, mức thống trị

In [ ]:
CAT_ALL = NOMINAL + ORDINAL_CAT
card = pd.DataFrame({
    "n_levels": [df_cat[c].nunique() for c in CAT_ALL],
    "top_level": [df_cat[c].value_counts().index[0] for c in CAT_ALL],
    "top_%": [round(100 * df_cat[c].value_counts(normalize=True).iloc[0], 1) for c in CAT_ALL],
    "n_rare(<1%)": [int((df_cat[c].value_counts(normalize=True) < 0.01).sum()) for c in CAT_ALL],
}, index=CAT_ALL)
print("Biến gần như hằng số (một mức chiếm ≥ 90%):")
card[card["top_%"] >= 90].sort_values("top_%", ascending=False)

In [ ]:
n, nc = len(NOMINAL), 4
nr = int(np.ceil(n / nc))
fig, axes = plt.subplots(nr, nc, figsize=(16, 3.2 * nr)); axes = axes.ravel()
for a, c in zip(axes, NOMINAL):
    vc = df_cat[c].astype(str).value_counts().head(10)
    sns.barplot(x=vc.values, y=vc.index, ax=a, color="#8172B2"); a.set_title(f"{c} ({df_cat[c].nunique()} mức)"); a.set_ylabel("")
for a in axes[n:]: a.axis("off")
save("04_nominal_bars"); plt.show()

In [ ]:
n, nc = len(ORDINAL_CAT), 4
nr = int(np.ceil(n / nc))
fig, axes = plt.subplots(nr, nc, figsize=(16, 3.0 * nr)); axes = axes.ravel()
for a, c in zip(axes, ORDINAL_CAT):
    order = sorted(ORD_MAPS[c], key=ORD_MAPS[c].get)
    sns.countplot(x=df_cat[c], order=[o for o in order if o in set(df_cat[c])], ax=a, color="#CCB974"); a.set_title(c); a.set_xlabel("")
for a in axes[n:]: a.axis("off")
save("05_ordinal_bars"); plt.show()

## 3. Phân tích đa biến
### 3.1 Tương quan với `SalePrice` (Pearson & Spearman)

In [ ]:
COR_COLS = CONTINUOUS + DISCRETE + ORDINAL_NUM + ORDINAL_CAT
cor_t = pd.DataFrame({
    "pearson": df_enc[COR_COLS].corrwith(df_enc[TARGET]),
    "spearman": df_enc[COR_COLS].corrwith(df_enc[TARGET], method="spearman"),
}).sort_values("pearson", ascending=False)
cor_t["|gap|"] = (cor_t["pearson"] - cor_t["spearman"]).abs()
print("Top 15 tương quan dương / 5 thấp nhất:")
display_tbl = pd.concat([cor_t.head(15), cor_t.tail(5)]).round(3)
display_tbl

In [ ]:
top = cor_t["pearson"].abs().sort_values(ascending=False).head(15).index
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
cor_t.loc[top, "pearson"].sort_values().plot.barh(ax=ax[0], color="#4C72B0"); ax[0].set_title("Top 15 |Pearson| với SalePrice")
cm_cols = list(cor_t["pearson"].abs().sort_values(ascending=False).head(10).index) + [TARGET]
sns.heatmap(df_enc[cm_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, ax=ax[1], cbar=False)
ax[1].set_title("Heatmap tương quan – 10 biến mạnh nhất")
save("06_correlation_target"); plt.show()

### 3.2 Đa cộng tuyến: các cặp biến có |r| > 0.8

In [ ]:
corr_all = df_enc[COR_COLS + [TARGET]].corr()
c = corr_all.abs()
up = c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
pairs = up.stack().reset_index(); pairs.columns = ["var1", "var2", "|r|"]
pairs = pairs[(pairs["|r|"] > 0.8) & (~pairs[["var1", "var2"]].isin([TARGET]).any(axis=1))].sort_values("|r|", ascending=False)
pairs.round(3).reset_index(drop=True)

### 3.3 Quan hệ biến định lượng – `SalePrice` (kèm đánh dấu điểm nghi ngờ GrLivArea > 4000)

In [ ]:
sel = ["GrLivArea", "TotalBsmtSF", "GarageArea", "1stFlrSF", "YearBuilt", "LotArea"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.ravel()
hi = df["GrLivArea"] > 4000
for a, c in zip(axes, sel):
    a.scatter(df[c], df[TARGET], s=10, alpha=0.4, color="#4C72B0")
    a.scatter(df.loc[hi, c], df.loc[hi, TARGET], s=40, color="red", label="GrLivArea > 4000")
    sns.regplot(x=df[c], y=df[TARGET], scatter=False, ax=a, color="black", line_kws={"lw": 1.2})
    a.set_title(f"{c} vs SalePrice (r={df[c].corr(df[TARGET]):.2f})")
axes[0].legend()
save("07_scatter_top_numeric"); plt.show()

### 3.4 Biến phân loại – `SalePrice` (Boxplot) và mức độ liên hệ (η², Kruskal–Wallis)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 10))
sns.boxplot(data=df, x="OverallQual", y=TARGET, ax=ax[0, 0], color="#4C72B0"); ax[0, 0].set_title("SalePrice theo OverallQual")
order = df.groupby("Neighborhood")[TARGET].median().sort_values().index
sns.boxplot(data=df, x="Neighborhood", y=TARGET, order=order, ax=ax[0, 1], color="#55A868")
ax[0, 1].tick_params(axis="x", rotation=90); ax[0, 1].set_title("SalePrice theo Neighborhood (sắp theo median)")
sns.boxplot(data=df, x="SaleCondition", y=TARGET, ax=ax[1, 0], color="#C44E52"); ax[1, 0].set_title("SalePrice theo SaleCondition")
sns.boxplot(data=df, x="MSZoning", y=TARGET, ax=ax[1, 1], color="#8172B2"); ax[1, 1].set_title("SalePrice theo MSZoning")
save("08_boxplots_categorical"); plt.show()

In [ ]:
def eta_sq(frame, cat, y=TARGET):
    grand = frame[y].mean()
    ssb = sum(len(g) * (g.mean() - grand) ** 2 for _, g in frame.groupby(cat)[y])
    return ssb / ((frame[y] - grand) ** 2).sum()

rows = []
for c in CAT_ALL:
    groups = [g[TARGET].values for _, g in df_cat.groupby(c)]
    p = stats.kruskal(*groups)[1] if len(groups) > 1 else np.nan
    rows.append((c, df_cat[c].nunique(), eta_sq(df_cat, c), p))
assoc = pd.DataFrame(rows, columns=["biến", "n_levels", "eta²", "p_kruskal"]).set_index("biến").sort_values("eta²", ascending=False)
fig, ax = plt.subplots(figsize=(9, 6))
assoc["eta²"].head(15).sort_values().plot.barh(ax=ax, color="#8172B2")
ax.set_title("Top 15 biến phân loại giải thích biến thiên SalePrice (η²)")
save("09_eta_squared"); plt.show()
assoc.head(15).round(4)

### 3.5 Tái hiện Hình 4 & Hình 7 của paper: tổng diện tích (TotalSF = TotalBsmtSF + GrLivArea) và SaleCondition

In [ ]:
df["TotalSF"] = df["TotalBsmtSF"] + df["GrLivArea"]
lr = stats.linregress(df["TotalSF"], df[TARGET])
print(f"SalePrice = {lr.intercept:.0f} + {lr.slope:.2f}·TotalSF | R² = {lr.rvalue**2:.3f}")
lr2 = stats.linregress(df.loc[~hi, "TotalSF"], df.loc[~hi, TARGET])
print(f"Bỏ GrLivArea>4000: R² = {lr2.rvalue**2:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=df, x="TotalSF", y=TARGET, hue="SaleCondition", alpha=0.7, s=25, ax=ax[0]); ax[0].set_title("SalePrice vs TotalSF theo SaleCondition (paper – Fig. 7)")
sns.scatterplot(data=df, x="TotalSF", y=np.log(df[TARGET]), hue="SaleCondition", alpha=0.7, s=25, ax=ax[1]); ax[1].set_title("Cùng biểu đồ với log(SalePrice) – phương sai ổn định hơn")
save("10_totalsf_salecondition"); plt.show()

### 3.6 Tính chất theo thời gian (2006–2010)

In [ ]:
ts = df.groupby(["YrSold", "MoSold"])[TARGET].agg(["median", "count"]).reset_index()
ts["date"] = pd.to_datetime(dict(year=ts.YrSold, month=ts.MoSold, day=1))
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(ts["date"], ts["median"], marker="o", color="#4C72B0"); ax.set_ylabel("Median SalePrice")
ax2 = ax.twinx(); ax2.bar(ts["date"], ts["count"], width=20, alpha=0.25, color="grey"); ax2.set_ylabel("Số giao dịch"); ax2.grid(False)
ax.set_title("Giá trung vị và số giao dịch theo tháng")
save("11_time_trend"); plt.show()
print("Median theo năm:\n", df.groupby("YrSold")[TARGET].median().to_string())

## 4. Giá trị thiếu & mẫu bất thường
### 4.1 Thống kê giá trị thiếu

In [ ]:
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss_tbl = pd.DataFrame({"số_thiếu": miss, "%": (miss / len(df) * 100).round(2), "kiểu": df[miss.index].dtypes.astype(str)})
print(f"{len(miss)} / {df.shape[1]} cột có NaN; tổng ô thiếu = {int(miss.sum())} ({100*miss.sum()/df.size:.2f}% toàn bảng)")
fig, ax = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.4]})
sns.barplot(x=miss_tbl["%"], y=miss_tbl.index, ax=ax[0], color="#C44E52"); ax[0].set_title("% thiếu theo cột")
sns.heatmap(df[miss.index].isna().T, cbar=False, ax=ax[1], xticklabels=False, cmap="Greys"); ax[1].set_title("Bản đồ thiếu (vạch đen = thiếu)")
save("12_missing_overview"); plt.show()
miss_tbl

### 4.2 Phân biệt "thiếu cấu trúc" (NA = không có tiện ích) và "thiếu thật"
Một cột là *thiếu cấu trúc* nếu NaN trùng khớp với điều kiện "không có" (vd. `PoolArea == 0`).

In [ ]:
STRUCT = {"PoolQC": ("PoolArea==0", df.PoolArea == 0), "FireplaceQu": ("Fireplaces==0", df.Fireplaces == 0),
          "MiscFeature": ("MiscVal==0", df.MiscVal == 0)}
for c in ["GarageType", "GarageYrBlt", "GarageFinish", "GarageQual", "GarageCond"]:
    STRUCT[c] = ("GarageArea==0", df.GarageArea == 0)
for c in ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]:
    STRUCT[c] = ("TotalBsmtSF==0", df.TotalBsmtSF == 0)
rows = []
for c, (cond_txt, cond) in STRUCT.items():
    na = df[c].isna()
    rows.append((c, int(na.sum()), int((na & cond).sum()), int((na & ~cond).sum()), cond_txt))
struct_tbl = pd.DataFrame(rows, columns=["cột", "n_NaN", "NaN khớp 'không có'", "NaN mâu thuẫn", "điều kiện"]).set_index("cột")
print("→ NaN mâu thuẫn > 0 = nghi lỗi nhập liệu/thiếu thật cần xử lý riêng:")
struct_tbl

In [ ]:
print("Alley, Fence: không có biến đi kèm; mã gốc ghi NA = 'No alley access' / 'No fence' → thiếu cấu trúc.")
print("NaN cùng dòng MasVnrType & MasVnrArea:", bool((df.MasVnrType.isna() == df.MasVnrArea.isna()).all()))
print("Electrical thiếu:", int(df.Electrical.isna().sum()), "dòng (thiếu thật, chọn mode)")
print("\nBất nhất MasVnr: Type='None' nhưng Area>0 →", int(((df.MasVnrType == "None") & (df.MasVnrArea > 0)).sum()),
      "| Type có nhưng Area==0 →", int(((df.MasVnrType.notna()) & (df.MasVnrType != "None") & (df.MasVnrArea == 0)).sum()))

# LotFrontage – thiếu thật nhiều nhất
lf = df["LotFrontage"]
print(f"\nLotFrontage thiếu {lf.isna().mean()*100:.1f}%. corr(LotFrontage, sqrt(LotArea)) = {lf.corr(np.sqrt(df.LotArea)):.2f}")
u = stats.mannwhitneyu(df.loc[lf.isna(), TARGET], df.loc[lf.notna(), TARGET])
print(f"Kiểm định thiếu có liên quan giá? Mann–Whitney p = {u.pvalue:.3f} (p nhỏ → thiếu không hoàn toàn ngẫu nhiên)")
by_nb = df.groupby("Neighborhood")["LotFrontage"].agg(["count", "median"]).assign(pct_NaN=df.groupby("Neighborhood")["LotFrontage"].apply(lambda s: s.isna().mean() * 100)).round(1)
by_nb.sort_values("pct_NaN", ascending=False).head(8)

**Chiến lược xử lý đề xuất** (áp dụng ở mục 6): thiếu cấu trúc → mức `"None"` (+ cờ `HasGarage`, `HasBsmt`…); `MasVnrType/Area` → `"None"`/0; `Electrical` → mode; `LotFrontage` → median theo `Neighborhood`.

### 4.3 Mẫu bất thường – kiểm tra tính nhất quán logic

In [ ]:
chk = {
    "YearBuilt > YrSold": df.YearBuilt > df.YrSold,
    "YearRemodAdd > YrSold": df.YearRemodAdd > df.YrSold,
    "YearRemodAdd < YearBuilt": df.YearRemodAdd < df.YearBuilt,
    "GarageYrBlt > YrSold": df.GarageYrBlt > df.YrSold,
    "GarageYrBlt < YearBuilt": df.GarageYrBlt < df.YearBuilt,
    "GrLivArea ≠ 1stFlr+2ndFlr+LowQual": df.GrLivArea != df["1stFlrSF"] + df["2ndFlrSF"] + df.LowQualFinSF,
    "TotalBsmtSF ≠ BsmtFin1+Fin2+Unf": df.TotalBsmtSF != df.BsmtFinSF1 + df.BsmtFinSF2 + df.BsmtUnfSF,
    "BedroomAbvGr > TotRmsAbvGrd": df.BedroomAbvGr > df.TotRmsAbvGrd,
    "PoolArea>0 nhưng PoolQC NaN": (df.PoolArea > 0) & df.PoolQC.isna(),
    "PoolArea==0 nhưng có PoolQC": (df.PoolArea == 0) & df.PoolQC.notna(),
    "GarageCars==0 nhưng GarageArea>0": (df.GarageCars == 0) & (df.GarageArea > 0),
    "Fireplaces==0 nhưng có FireplaceQu": (df.Fireplaces == 0) & df.FireplaceQu.notna(),
    "MiscVal>0 nhưng MiscFeature NaN": (df.MiscVal > 0) & df.MiscFeature.isna(),
    "2ndFlrSF>0 nhưng HouseStyle='1Story'": (df["2ndFlrSF"] > 0) & (df.HouseStyle == "1Story"),
    "Giá/GrLivArea < 30 hoặc > 300 USD/sqft": (df[TARGET] / df.GrLivArea < 30) | (df[TARGET] / df.GrLivArea > 300),
}
anom = pd.DataFrame({"số dòng": {k: int(v.sum()) for k, v in chk.items()},
                     "Id (tối đa 8)": {k: ", ".join(map(str, df.loc[v, "Id"].head(8))) for k, v in chk.items()}})
anom

In [ ]:
print("Nhãn không đồng nhất giữa Exterior1st và Exterior2nd (khác chính tả cùng một chất liệu):")
print("  chỉ có ở Exterior2nd:", sorted(set(df.Exterior2nd.dropna()) - set(df.Exterior1st.dropna())))
print("  chỉ có ở Exterior1st:", sorted(set(df.Exterior1st.dropna()) - set(df.Exterior2nd.dropna())))
print("\nGiao dịch không điển hình (paper: Partial/Abnormal/Family… có giá khác 'Normal'):")
print(df["SaleCondition"].value_counts().to_string())
print((df.groupby("SaleCondition")[TARGET].median()).round(0).to_string())

## 5. Xác định ngoại lệ (outliers)
### 5.1 Đơn biến – IQR (1.5×IQR và 3×IQR) & z-score

In [ ]:
def iqr_out(s, k):
    s = s.dropna(); q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - k * i) | (s > q3 + k * i)).sum())
rows = []
for c in CONTINUOUS + [TARGET]:
    s = df[c]
    rows.append((c, iqr_out(s, 1.5), iqr_out(s, 3.0), int((np.abs(stats.zscore(s.dropna())) > 3).sum()), round(100 * (s == 0).mean(), 1)))
out_tbl = pd.DataFrame(rows, columns=["biến", "n_IQR1.5", "n_IQR3.0", "n_|z|>3", "%zero"]).set_index("biến").sort_values("n_IQR1.5", ascending=False)
print("Lưu ý: biến có %zero cao → IQR thu hẹp, số ngoại lệ bị thổi phồng.")
out_tbl

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7)); axes = axes.ravel()
for a, c in zip(axes, [TARGET, "LotArea", "GrLivArea", "TotalBsmtSF", "1stFlrSF", "GarageArea", "LotFrontage", "MasVnrArea"]):
    sns.boxplot(x=df[c], ax=a, color="#64B5CD", fliersize=2); a.set_title(c); a.set_xlabel("")
save("13_boxplots_outliers"); plt.show()

### 5.2 Hai biến – điểm lệch khỏi xu hướng (paper khuyến nghị bỏ GrLivArea > 4000)

In [ ]:
thr = {"GrLivArea": 4000, "TotalBsmtSF": 5000, "1stFlrSF": 4000, "LotArea": 100000, "LotFrontage": 300, "BsmtFinSF1": 4000}
for c, t in thr.items():
    sub = df.loc[df[c] > t, ["Id", c, TARGET, "OverallQual", "SaleCondition"]]
    print(f"\n{c} > {t}: {len(sub)} dòng"); print(sub.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, yy, lab in [(ax[0], df[TARGET], "SalePrice"), (ax[1], np.log(df[TARGET]), "log(SalePrice)")]:
    a.scatter(df.GrLivArea, yy, s=12, alpha=0.4); a.scatter(df.loc[hi, "GrLivArea"], yy[hi], c="red", s=50)
    for _, r in df[hi].iterrows():
        a.annotate(f"Id {int(r.Id)}\n{r.SaleCondition}", (r.GrLivArea, np.log(r[TARGET]) if lab.startswith("log") else r[TARGET]), fontsize=8, xytext=(5, 5), textcoords="offset points")
    a.set_xlabel("GrLivArea"); a.set_ylabel(lab); a.set_title(f"GrLivArea vs {lab}")
save("14_grlivarea_outliers"); plt.show()

### 5.3 Đa biến – Cook's distance & phần dư chuẩn hoá (mô hình OLS đơn giản trên log(SalePrice))

In [ ]:
feat = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt"]
X = np.column_stack([np.ones(len(df)), df[feat].values.astype(float)])
yl = np.log(df[TARGET].values)
beta, *_ = np.linalg.lstsq(X, yl, rcond=None)
fitted = X @ beta; resid = yl - fitted
n_obs, p = X.shape
h = np.einsum("ij,ji->i", X, np.linalg.pinv(X.T @ X) @ X.T)       # leverage (đường chéo ma trận hat)
s2 = resid @ resid / (n_obs - p)
cooks = resid**2 / (p * s2) * h / (1 - h) ** 2                     # Cook's distance
rstud = resid / np.sqrt(s2 * (1 - h))                              # phần dư chuẩn hoá (internally studentized)
r2 = 1 - resid @ resid / ((yl - yl.mean()) ** 2).sum()
df["cooks"], df["rstud"] = cooks, rstud
th = 4 / len(df)
print(f"R² mô hình thăm dò = {r2:.3f}")
print(f"Cook's D > 4/n ({th:.4f}): {(cooks > th).sum()} điểm | |studentized resid| > 3: {(np.abs(rstud) > 3).sum()} điểm")
print("\nTop 10 điểm ảnh hưởng nhất:")
print(df.nlargest(10, "cooks")[["Id", "GrLivArea", "OverallQual", TARGET, "SaleCondition", "cooks", "rstud"]].round(3).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].stem(df.index, cooks, markerfmt=",", basefmt=" "); ax[0].axhline(th, c="r", ls="--", label="4/n"); ax[0].legend(); ax[0].set_title("Cook's distance")
ax[1].scatter(fitted, rstud, s=10, alpha=0.5); ax[1].axhline(3, c="r", ls="--"); ax[1].axhline(-3, c="r", ls="--")
ax[1].set_title("Phần dư chuẩn hoá vs giá trị dự đoán"); ax[1].set_xlabel("fitted log(price)")
save("15_cooks_residuals"); plt.show()

### 5.4 Quyết định xử lý & tác động
Theo paper, 2 căn *Partial* có diện tích rất lớn nhưng giá thấp là "true outliers" (không phản ánh giá thị trường); các căn lớn + giá cao là hợp lý → giữ.

In [ ]:
out_ids = df.loc[(df.GrLivArea > 4000) & (df[TARGET] < 300000), "Id"].tolist()
print("Id loại bỏ:", out_ids)
keep = ~df.Id.isin(out_ids)
for nm, m in [("Trước", pd.Series(True, index=df.index)), ("Sau", keep)]:
    s = stats.linregress(df.loc[m, "GrLivArea"], df.loc[m, TARGET])
    print(f"{nm}: n={m.sum()}, r(GrLivArea, SalePrice)={s.rvalue:.3f}, R²={s.rvalue**2:.3f}, hệ số góc={s.slope:.1f}")

## 6. Tổng kết & xuất dữ liệu đã làm sạch

In [ ]:
d = df.drop(columns=["cooks", "rstud", "TotalSF"]).copy()
cat_struct = [c for c in ["Alley", "Fence", "MiscFeature", "PoolQC", "FireplaceQu", "GarageType", "GarageFinish", "GarageQual",
                          "GarageCond", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]]
d["HasGarage"] = (d.GarageArea > 0).astype(int); d["HasBsmt"] = (d.TotalBsmtSF > 0).astype(int)
for c in cat_struct: d[c] = d[c].fillna("None")
d["GarageYrBlt"] = d["GarageYrBlt"].fillna(0)
d["MasVnrType"] = d["MasVnrType"].fillna("None"); d["MasVnrArea"] = d["MasVnrArea"].fillna(0)
d["Electrical"] = d["Electrical"].fillna(d["Electrical"].mode()[0])
d["LotFrontage"] = d["LotFrontage"].fillna(d.groupby("Neighborhood")["LotFrontage"].transform("median")).fillna(d["LotFrontage"].median())
d = d[~d.Id.isin(out_ids)].reset_index(drop=True)
print("Còn NaN sau xử lý:", int(d.isna().sum().sum()), "| kích thước:", d.shape)
d.to_csv("train_cleaned.csv", index=False)

print("\n===== TÓM TẮT PHÁT HIỆN CHÍNH =====")
print(f"1. SalePrice lệch phải (skew={y.skew():.2f}) → log1p đưa skew về {np.log1p(y).skew():.2f}.")
print("2. Top biến định lượng:", ", ".join(cor_t['pearson'].abs().sort_values(ascending=False).head(5).index))
print("3. Top biến phân loại (η²):", ", ".join(assoc.index[:5]))
print("4. Cặp đa cộng tuyến |r|>0.8:", "; ".join(f"{a}–{b}" for a, b in pairs[["var1", "var2"]].values) or "không có")
print(f"5. {len(miss)} cột có NaN; phần lớn là thiếu cấu trúc; thiếu thật: LotFrontage, MasVnrType/Area, Electrical.")
print("6. Ngoại lệ chính: Id", out_ids, "(GrLivArea>4000, giá thấp).")